# Técnicas de Diseño y Validación de Modelos
## Evaluación Integradora — Parte 1

<div style="display:flex; align-items:center; gap:12px; margin: 8px 0 18px 0;">
  <a href="https://colab.research.google.com/github/GabrielaOjcius/Laboratorios-Tecnicas-Diseno-Validacion-Modelos/blob/main/Evaluacion%20integradora/F_Evaluacion%20Integradora%20-%20Parte%201%20-%20Notebook.ipynb" target="_blank">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/>
  </a>
  <span>En Colab, antes de editar: <b>Archivo → Guardar una copia en Drive</b>.</span>
</div>

**Caso:** predicción de fallas en máquinas a partir de sensores.  
**Dataset:** AI4I 2020 Predictive Maintenance Dataset — UCI Machine Learning Repository.  
**Ejes evaluados:** comprensión del problema, desbalance de clases, fuga de información, partición de datos y comparación inicial de modelos.

---

### Propósito de esta parte

En esta primera parte analizás el problema de mantenimiento predictivo y comparás modelos supervisados mediante validación cruzada. El foco está en entender por qué la exactitud puede ser engañosa cuando la clase positiva es poco frecuente, y por qué deben excluirse columnas que filtran directamente la variable respuesta antes de que ocurra el evento.

### Instrucciones

1. Ejecutá las celdas en orden, sin saltear ninguna.
2. Completá las respuestas en las celdas marcadas con **✏️ Respuesta**.
3. Justificá con valores numéricos obtenidos de las tablas y gráficos.
4. No utilizés el conjunto de test en esta parte para elegir el modelo final.
5. Entregá el notebook con todos los outputs visibles.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    make_scorer,
)
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")

---
## A. Carga y exploración inicial de los datos

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
df = pd.read_csv(url)
df.head()

In [ ]:
df.info()

In [ ]:
df["Machine failure"].value_counts(normalize=True).rename("proporción")

**Pregunta A.1**  
¿El problema está balanceado o desbalanceado? ¿Por qué esto afecta la interpretación de la exactitud (`accuracy`)?

✏️ **Respuesta:**  
*(Escribí tu respuesta aquí)*

---
## B. Preparación del dataset

Las columnas `TWF`, `HDF`, `PWF`, `OSF` y `RNF` describen **tipos específicos de falla**. Como están directamente ligadas al resultado `Machine failure`, no pueden usarse como variables predictoras: en un escenario real, esta información no estaría disponible en el momento en que debería hacerse la predicción. Incluirlas equivale a permitir que el modelo vea la respuesta antes de predecirla, lo que constituye **fuga de información** (*data leakage*).

In [ ]:
target = "Machine failure"
columnas_fuga = ["TWF", "HDF", "PWF", "OSF", "RNF"]
columnas_descartar = ["UDI", "Product ID", target] + columnas_fuga

X = df.drop(columns=columnas_descartar)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

categoricas = X.select_dtypes(include="object").columns.tolist()
numericas   = X.select_dtypes(exclude="object").columns.tolist()

preprocesamiento = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas),
        ("num", StandardScaler(), numericas),
    ]
)

print("Tamaños:")
print(f"  Train: {X_train.shape[0]} muestras  |  Tasa de falla: {y_train.mean():.2%}")
print(f"  Test:  {X_test.shape[0]}  muestras  |  Tasa de falla: {y_test.mean():.2%}")
print(f"  Variables numéricas: {numericas}")
print(f"  Variables categóricas: {categoricas}")

**Pregunta B.1**  
Explicá por qué se descartan las columnas de tipos de falla antes de entrenar los modelos. ¿Qué pasaría si las usáramos como variables predictoras?

✏️ **Respuesta:**  
*(Escribí tu respuesta aquí)*

---
## C. Comparación de modelos mediante validación cruzada

Compararemos un modelo trivial con varios modelos supervisados. En problemas desbalanceados, el modelo trivial cumple una función específica: si un modelo complejo no supera esa referencia mínima en las métricas relevantes, entonces no aporta ningún valor respecto a no hacer nada.

In [ ]:
modelos = {
    "Dummy (mayoría)": DummyClassifier(strategy="most_frequent"),
    "Regresión Logística": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Árbol de Decisión": DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_leaf=5,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

scoring = {
    "accuracy":           "accuracy",
    "recall":             make_scorer(recall_score, zero_division=0),
    "precision":          make_scorer(precision_score, zero_division=0),
    "roc_auc":            "roc_auc",
    "average_precision":  "average_precision",
}

filas = []
for nombre, modelo in modelos.items():
    pipe = modelo if nombre == "Dummy (mayoría)" else Pipeline([("prep", preprocesamiento), ("model", modelo)])
    scores = cross_validate(pipe, X_train, y_train, cv=5, scoring=scoring, n_jobs=-1)
    filas.append({
        "modelo":               nombre,
        "accuracy_cv":          scores["test_accuracy"].mean(),
        "recall_cv":            scores["test_recall"].mean(),
        "precision_cv":         scores["test_precision"].mean(),
        "roc_auc_cv":           scores["test_roc_auc"].mean(),
        "average_precision_cv": scores["test_average_precision"].mean(),
    })

tabla_modelos = pd.DataFrame(filas).set_index("modelo").sort_values("average_precision_cv", ascending=False)
tabla_modelos.style.highlight_max(axis=0, color="#d4edda")

In [ ]:
metricas_viz = ["recall_cv", "roc_auc_cv", "average_precision_cv"]
etiquetas    = ["Recall", "ROC-AUC", "Average Precision"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colores = ["#8B1A2F", "#2E75B6", "#28A745", "#F4A300", "#6C3483"]

for ax, metrica, etiqueta in zip(axes, metricas_viz, etiquetas):
    vals = tabla_modelos[metrica]
    bars = ax.barh(vals.index, vals.values, color=colores[:len(vals)])
    ax.set_xlabel(etiqueta)
    ax.set_title(etiqueta, fontweight="bold")
    ax.set_xlim(0, 1.05)
    for bar, v in zip(bars, vals.values):
        ax.text(v + 0.01, bar.get_y() + bar.get_height() / 2, f"{v:.3f}", va="center", fontsize=9)

plt.suptitle("Comparación de modelos — Validación cruzada (5 pliegues)", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

**Pregunta C.1**  
¿Qué modelo elegirías a partir de los resultados de la validación cruzada? Justificá tu elección usando al menos dos métricas de la tabla.

✏️ **Respuesta:**  
*(Escribí tu respuesta aquí)*

---

**Pregunta C.2**  
¿Por qué el modelo trivial puede tener una exactitud alta aunque no detecte fallas? ¿Qué revela esto sobre el uso de `accuracy` como métrica única en este problema?

✏️ **Respuesta:**  
*(Escribí tu respuesta aquí)*

---
## D. Análisis de la curva de aprendizaje del modelo seleccionado

La curva de aprendizaje permite diagnosticar si el modelo tiene problemas de sesgo (underfitting) o varianza (overfitting) según el tamaño del conjunto de entrenamiento. Este análisis corresponde a los contenidos del Módulo 3.

In [ ]:
from sklearn.model_selection import learning_curve, StratifiedKFold

pipe_rf = Pipeline([
    ("prep", preprocesamiento),
    ("model", RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_leaf=5,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
    )),
])

cv_lc = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

train_sizes, train_scores, val_scores = learning_curve(
    pipe_rf, X_train, y_train,
    cv=cv_lc,
    scoring="average_precision",
    train_sizes=np.linspace(0.15, 1.0, 8),
    shuffle=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_mean, "o-", color="#8B1A2F", label="Average Precision — entrenamiento", lw=2)
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.12, color="#8B1A2F")
ax.plot(train_sizes, val_mean, "s--", color="#2E75B6", label="Average Precision — validación CV", lw=2)
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.12, color="#2E75B6")
ax.set_xlabel("Tamaño del conjunto de entrenamiento")
ax.set_ylabel("Average Precision")
ax.set_title("Curva de aprendizaje — Random Forest", fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Pregunta D.1**  
Observá la curva de aprendizaje. ¿La brecha entre entrenamiento y validación disminuye al aumentar el tamaño del conjunto de entrenamiento? ¿Qué diagnóstico harías sobre el comportamiento del modelo (sobreajuste, subajuste, o ajuste razonable)?

✏️ **Respuesta:**  
*(Escribí tu respuesta aquí)*

---
## E. Cierre de la Parte 1

Redactá una conclusión de entre **150 y 250 palabras** que sintetice lo que aprendiste sobre desbalance de clases, fuga de información y selección preliminar de modelos. Incluí al menos dos valores numéricos de los resultados obtenidos para sostener tus afirmaciones.

✏️ **Conclusión:**  
*(Escribí tu conclusión aquí)*

---
*Técnicas de Diseño y Validación de Modelos (0204) — Universidad Blas Pascal*